# Pixel-Domain vs. Frequency-Domain (2D-DCT) CNNs for Pneumonia Detection

A controlled comparison of two image representations - raw **pixels** vs. **2D-DCT frequency coefficients** - fed to *identical* CNNs to detect pneumonia in chest X-rays.

This mirrors the methodology of my undergraduate thesis (pixel- vs. frequency-domain 2D-DCT features for StyleGAN deepfake detection), transferred to a medical-imaging domain.

**Author:** Steven Jerry Gani | **GitHub:** @Handcull

> Disclaimer: research/educational project only - not a medical device, not for clinical use.

## 1. Environment setup

Set the runtime to GPU: **Runtime -> Change runtime type -> GPU**. The install below pins the `datasets` library to a version that still supports this dataset. If the import in the next section fails right after it, do **Runtime -> Restart session** and run all again (a one-time downgrade hiccup).

In [ ]:
# Colab already ships TensorFlow, NumPy, SciPy, scikit-learn, matplotlib, seaborn, pandas.
# This dataset loads via a Python script, which datasets 4.x removed - so we pin datasets < 4.
!pip -q install 'datasets>=2.18,<4'

In [ ]:
import os, random
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.fft import dctn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay)
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
sns.set_theme(style='whitegrid')

print('TensorFlow', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU available:', bool(gpus), gpus)

## 2. Load the dataset (streamed from the cloud)

The data is pulled directly into the Colab runtime from HuggingFace. It lands in Colab's cloud cache - **never on your local PC**. Only this notebook's text lives on your machine.

In [ ]:
from datasets import load_dataset

DATASET = 'keremberke/chest-xray-classification'
# This dataset loads via a Python script, so trust_remote_code=True is required.
try:
    ds = load_dataset(DATASET, name='full', trust_remote_code=True)
except TypeError:
    ds = load_dataset(DATASET, name='full')
print(ds)

class_names = ds['train'].features['labels'].names  # ['NORMAL', 'PNEUMONIA']
print('Classes:', class_names)

## 3. Preprocess into arrays

Convert each X-ray to grayscale, resize to 128x128, and scale pixels to [0, 1].

In [ ]:
IMG_SIZE = 128

def split_to_arrays(split):
    n = len(split)
    X = np.zeros((n, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    y = np.zeros((n,), dtype=np.int32)
    for i, ex in enumerate(split):
        img = ex['image'].convert('L').resize((IMG_SIZE, IMG_SIZE))
        X[i] = np.asarray(img, dtype=np.float32) / 255.0
        y[i] = ex['labels']
    return X, y

X_train, y_train = split_to_arrays(ds['train'])
X_val,   y_val   = split_to_arrays(ds['validation'])
X_test,  y_test  = split_to_arrays(ds['test'])

print('train', X_train.shape, '| val', X_val.shape, '| test', X_test.shape)

## 4. Exploratory look

In [ ]:
for name, y in [('train', y_train), ('val', y_val), ('test', y_test)]:
    counts = np.bincount(y, minlength=2)
    print(name, dict(zip(class_names, counts)))

plt.figure(figsize=(5, 3))
sns.barplot(x=class_names, y=np.bincount(y_train, minlength=2))
plt.title('Training class distribution'); plt.ylabel('count'); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for row, cls in enumerate([0, 1]):
    idx = np.where(y_train == cls)[0][:5]
    for col, j in enumerate(idx):
        axes[row, col].imshow(X_train[j], cmap='gray')
        axes[row, col].set_title(class_names[cls]); axes[row, col].axis('off')
plt.suptitle('Sample chest X-rays'); plt.tight_layout(); plt.show()

## 5. The frequency domain: the 2D-DCT

The 2D Discrete Cosine Transform expresses an image as a sum of cosine waves of increasing frequency. Low frequencies (smooth structure) concentrate in the top-left; high frequencies (fine texture, edges) spread toward the bottom-right.

For each image we compute  F = log(1 + |DCT(x)|)  and normalize to [0, 1] - this is the input to the DCT-CNN. The research question, exactly as in my thesis: does this frequency view make the target easier for a CNN to learn?

In [ ]:
def dct2_logmag(img):
    # 2D type-II DCT (orthonormal), then log-magnitude spectrum, normalized to [0, 1].
    D = dctn(img, type=2, norm='ortho')
    F = np.log1p(np.abs(D))
    F -= F.min()
    mx = F.max()
    if mx > 0:
        F /= mx
    return F.astype(np.float32)

fig, axes = plt.subplots(2, 2, figsize=(9, 9))
for row, cls in enumerate([0, 1]):
    j = np.where(y_train == cls)[0][0]
    axes[row, 0].imshow(X_train[j], cmap='gray')
    axes[row, 0].set_title(class_names[cls] + ' - pixels'); axes[row, 0].axis('off')
    axes[row, 1].imshow(dct2_logmag(X_train[j]), cmap='magma')
    axes[row, 1].set_title(class_names[cls] + ' - 2D-DCT (log-magnitude)'); axes[row, 1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
def mean_spectrum(X, y, cls, k=300):
    idx = np.where(y == cls)[0][:k]
    acc = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float64)
    for j in idx:
        acc += dct2_logmag(X[j])
    return acc / len(idx)

m0 = mean_spectrum(X_train, y_train, 0)
m1 = mean_spectrum(X_train, y_train, 1)
diff = m1 - m0
lim = np.abs(diff).max()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(m0, cmap='magma'); axes[0].set_title('Mean DCT - NORMAL'); axes[0].axis('off')
axes[1].imshow(m1, cmap='magma'); axes[1].set_title('Mean DCT - PNEUMONIA'); axes[1].axis('off')
im = axes[2].imshow(diff, cmap='seismic', vmin=-lim, vmax=lim)
axes[2].set_title('Difference (PNEUMONIA - NORMAL)'); axes[2].axis('off')
fig.colorbar(im, ax=axes[2], fraction=0.046)
plt.tight_layout(); plt.show()

## 6. Build the DCT feature set

Transform every split into its frequency representation. These arrays feed the DCT-CNN.

In [ ]:
def to_dct_set(X):
    out = np.zeros_like(X)
    for i in range(X.shape[0]):
        out[i] = dct2_logmag(X[i])
    return out

Xdct_train = to_dct_set(X_train)
Xdct_val   = to_dct_set(X_val)
Xdct_test  = to_dct_set(X_test)
print('DCT feature sets ready:', Xdct_train.shape)

## 7. Model (identical architecture for both branches)

Both models share the *same* architecture, optimizer, and training budget. The only difference is the input representation (pixels vs. DCT), so any performance gap is attributable to the representation.

In [ ]:
def build_cnn(input_shape, name='cnn'):
    inp = tf.keras.Input(shape=input_shape, name='input')
    x = inp
    for f in [16, 32, 64]:
        x = tf.keras.layers.Conv2D(f, 3, padding='same', use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation('relu')(x)
        x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(128, 3, padding='same', use_bias=False, name='last_conv')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(inp, out, name=name)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy',
                  metrics=['accuracy',
                           tf.keras.metrics.AUC(name='auc'),
                           tf.keras.metrics.Precision(name='precision'),
                           tf.keras.metrics.Recall(name='recall')])
    return model

cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight = {0: float(cw[0]), 1: float(cw[1])}
print('class weights:', class_weight)

build_cnn((IMG_SIZE, IMG_SIZE, 1)).summary()

## 8. Train - pixel-domain CNN

In [ ]:
def make_callbacks():
    return [
        tf.keras.callbacks.EarlyStopping(monitor='val_auc', mode='max',
                                         patience=6, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                             patience=3, min_lr=1e-5),
    ]

EPOCHS = 30
BATCH = 32

pixel_model = build_cnn((IMG_SIZE, IMG_SIZE, 1), name='pixel_cnn')
hist_pixel = pixel_model.fit(
    X_train[..., None], y_train,
    validation_data=(X_val[..., None], y_val),
    epochs=EPOCHS, batch_size=BATCH,
    class_weight=class_weight, callbacks=make_callbacks(), verbose=2)

## 9. Train - 2D-DCT CNN

In [ ]:
dct_model = build_cnn((IMG_SIZE, IMG_SIZE, 1), name='dct_cnn')
hist_dct = dct_model.fit(
    Xdct_train[..., None], y_train,
    validation_data=(Xdct_val[..., None], y_val),
    epochs=EPOCHS, batch_size=BATCH,
    class_weight=class_weight, callbacks=make_callbacks(), verbose=2)

## 10. Evaluate and compare

Recall matters most here: a false negative means missing a pneumonia case.

In [ ]:
def evaluate(model, X, y):
    p = model.predict(X[..., None], verbose=0).ravel()
    pred = (p >= 0.5).astype(int)
    metrics = {
        'accuracy': accuracy_score(y, pred),
        'precision': precision_score(y, pred),
        'recall': recall_score(y, pred),
        'f1': f1_score(y, pred),
        'roc_auc': roc_auc_score(y, p),
    }
    return metrics, p, pred

res_pixel, p_pixel, pred_pixel = evaluate(pixel_model, X_test, y_test)
res_dct,   p_dct,   pred_dct   = evaluate(dct_model,   Xdct_test, y_test)

table = pd.DataFrame({'Pixel-domain CNN': res_pixel, '2D-DCT CNN': res_dct}).T.round(4)
print(table)
table

In [ ]:
def plot_history(h, title):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(h.history['loss'], label='train'); ax[0].plot(h.history['val_loss'], label='val')
    ax[0].set_title(title + ' - loss'); ax[0].legend()
    ax[1].plot(h.history['auc'], label='train'); ax[1].plot(h.history['val_auc'], label='val')
    ax[1].set_title(title + ' - AUC'); ax[1].legend()
    plt.tight_layout(); plt.show()

plot_history(hist_pixel, 'Pixel-domain CNN')
plot_history(hist_dct, '2D-DCT CNN')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, pred, ttl in [(axes[0], pred_pixel, 'Pixel-domain CNN'),
                      (axes[1], pred_dct, '2D-DCT CNN')]:
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(ttl)
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
for p, lbl in [(p_pixel, 'Pixel-domain'), (p_dct, '2D-DCT')]:
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, label=lbl + ' (AUC=' + format(roc_auc_score(y_test, p), '.3f') + ')')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC - Pixel vs 2D-DCT'); plt.legend(); plt.show()

## 11. Grad-CAM: where does the model look?

Grad-CAM highlights the regions that most influenced the prediction - a key tool for *trustworthy AI* in medical imaging. We apply it to the pixel-domain model, where the heatmap maps back to anatomy. (For the DCT model the same map would live in frequency space and is harder to read.)

In [ ]:
def grad_cam(model, img, last_conv_name='last_conv'):
    grad_model = tf.keras.models.Model(
        model.inputs, [model.get_layer(last_conv_name).output, model.output])
    x = img[None, ..., None].astype('float32')
    with tf.GradientTape() as tape:
        conv_out, pred = grad_model(x)
        loss = pred[:, 0]
    grads = tape.gradient(loss, conv_out)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.reduce_sum(conv_out[0] * weights, axis=-1)
    cam = tf.nn.relu(cam)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE)).numpy().squeeze()
    return cam

idx = np.where(y_test == 1)[0][:4]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, j in zip(axes, idx):
    cam = grad_cam(pixel_model, X_test[j])
    prob = float(pixel_model.predict(X_test[j][None, ..., None], verbose=0).ravel()[0])
    ax.imshow(X_test[j], cmap='gray')
    ax.imshow(cam, cmap='jet', alpha=0.4)
    ax.set_title('p(pneumonia)=' + format(prob, '.2f')); ax.axis('off')
plt.suptitle('Grad-CAM - pixel-domain CNN'); plt.tight_layout(); plt.show()

## 12. Discussion and next steps

**What to report.** Fill the results table in the README with the test-set metrics above, and add the comparison plots from this notebook to `assets/`.

**Limitations.** The dataset is a re-export of the Kermany et al. chest-X-ray set; train/val/test come from the same distribution, so metrics may be optimistic. This is not validated for clinical use.

**Future work (ties back to the thesis).**
- Block-wise 8x8 DCT (JPEG-style) instead of full-image DCT.
- A two-stream network that fuses pixel and DCT features.
- Swap the small CNN for a transfer-learning backbone (e.g. EfficientNet).
- Repeat with k-fold cross-validation and report mean +/- std.

## 13. (Optional) Save artifacts back to GitHub

Easiest: **File -> Save a copy in GitHub** from the Colab menu.

To push exported figures into the repo's `assets/` folder, clone with a token, copy the PNGs in, then commit and push from a Colab cell.